In [1]:
import os
import re
import pandas as pd

# Jalur folder input dan output
RAW_TXT_FOLDER = os.path.abspath("data/raw")
PROCESSED_FOLDER = os.path.abspath("data/processed")
os.makedirs(PROCESSED_FOLDER, exist_ok=True)

def extract_metadata_and_features(file_path, case_id):
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()
    
    # 1. Ekstraksi Nomor Perkara
    # Pola biasa: nomor 123/pid.b/2024/pn mnd
    no_perkara_match = re.search(r"nomor\s+(\d+/pid\.[b|g].*?/pn\s+mnd)", text)
    no_perkara = no_perkara_match.group(1).upper() if no_perkara_match else "Tidak Terdeteksi"
    
    # 2. Ekstraksi Tanggal Putusan
    # Pola biasa: tanggal 10 maret 2025
    tanggal_match = re.search(r"tanggal\s+(\d+\s+[a-zA-Z]+\s+\d{4})", text)
    tanggal = tanggal_match.group(1) if tanggal_match else "Tidak Terdeteksi"
    
    # 3. Ekstraksi Nama Terdakwa / Pihak
    # Pola biasa: nama lengkap : thariq ...
    terdakwa_match = re.search(r"nama lengkap\s*:\s*([a-zA-Z\s]+?)(?:tempat|umur|lahir|$)", text)
    pihak = terdakwa_match.group(1).strip().upper() if terdakwa_match else "Tidak Terdeteksi"
    
    # 4. Ekstraksi Pasal yang Didakwakan (Fokus Kasus Pembunuhan)
    if "340" in text:
        pasal = "Pasal 340 KUHP (Pembunuhan Berencana)"
    elif "338" in text:
        pasal = "Pasal 338 KUHP (Pembunuhan Biasa)"
    else:
        pasal = "Pasal Pidana Lainnya"
        
    # 5. Ekstraksi Ringkasan Fakta (Mengambil potongan teks sekitar dakwaan)
    # Kita ambil 1000 karakter setelah kata 'menimbang bahwa' atau 'dakwaan' sebagai representasi fakta
    fakta_match = re.search(r"(bahwa ia terdakwa|menimbang bahwa).*?", text)
    ringkasan_fakta = text[fakta_match.start():fakta_match.start() + 1200] if fakta_match else text[:1200]
    
    # 6. Ekstraksi Amar Putusan / Solusi (Penting untuk Tahap Reuse)
    # Biasanya diawali kata 'mengadili' di akhir dokumen
    amar_match = re.search(r"mengadili\s*:\s*(.*)", text)
    amar_putusan = amar_match.group(1)[:1000] if amar_match else "Amar Putusan Tidak Tersegmentasi Otomatis"
    
    # Feature Engineering Sederhana: Hitung jumlah kata sesuai rubrik tugas
    text_length = len(text.split())
    
    return {
        "case_id": case_id,
        "no_perkara": no_perkara,
        "tanggal": tanggal,
        "pihak": pihak,
        "pasal": pasal,
        "ringkasan_fakta": ringkasan_fakta,
        "amar_putusan": amar_putusan,
        "text_length": text_length,
        "text_full": text
    }

def run_representation_pipeline():
    txt_files = [f for f in os.listdir(RAW_TXT_FOLDER) if f.endswith('.txt')]
    
    if not txt_files:
        print("Folder data raw kosong. Pastikan Tahap 1 sudah berjalan dengan benar.")
        return
        
    cases_data = []
    print(f"Mengekstrak fitur dari {len(txt_files)} file teks bersih...")
    
    for idx, filename in enumerate(sorted(txt_files)):
        file_path = os.path.join(RAW_TXT_FOLDER, filename)
        case_id = idx + 1
        
        # Jalankan ekstraksi per dokumen
        case_features = extract_metadata_and_features(file_path, case_id)
        cases_data.append(case_features)
        
    # Ubah menjadi Pandas DataFrame agar terstruktur
    df = pd.DataFrame(cases_data)
    
    # Simpan ke format CSV sesuai output yang diharapkan tugas
    csv_output_path = os.path.join(PROCESSED_FOLDER, "cases.csv")
    df.to_csv(csv_output_path, index=False)
    
    print(f"\n=======================================================")
    print(f"Selesai! Dataset terstruktur berhasil dibuat.")
    print(f"Disimpan di: {csv_output_path}")
    print(f"=======================================================")
    print(df[["case_id", "no_perkara", "pihak", "pasal", "text_length"]].head())

# Jalankan pipeline representation
run_representation_pipeline()

Mengekstrak fitur dari 40 file teks bersih...

Selesai! Dataset terstruktur berhasil dibuat.
Disimpan di: d:\Kuliah\Semester 6\Computer Reasoning\Sub-CPMK3\cbr-pembunuhan\notebooks\data\processed\cases.csv
   case_id             no_perkara             pihak  \
0        1       Tidak Terdeteksi  Tidak Terdeteksi   
1        2       Tidak Terdeteksi  Tidak Terdeteksi   
2        3  133/PID.B/2024/PN MND  Tidak Terdeteksi   
3        4       Tidak Terdeteksi  Tidak Terdeteksi   
4        5       Tidak Terdeteksi  Tidak Terdeteksi   

                                   pasal  text_length  
0      Pasal 338 KUHP (Pembunuhan Biasa)         8235  
1      Pasal 338 KUHP (Pembunuhan Biasa)         8235  
2      Pasal 338 KUHP (Pembunuhan Biasa)         3952  
3  Pasal 340 KUHP (Pembunuhan Berencana)        11584  
4                   Pasal Pidana Lainnya         3619  
